# optimizer-state-tensor-buffers composite — cx30: v EMA stored per-param in a state dict (PyTorch optimizer convention)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `ema-second-moment`, `optimizer-state-tensor-buffers`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "optimizer-state-tensor-buffers"
DD_ATOM_IDS = ["ema-second-moment", "optimizer-state-tensor-buffers"]
DD_SUBTOPICS = ["Optimizer: Adam EMA second moment", "Optimizer: Per-param state buffers"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`torch.optim.Optimizer` stores per-parameter state in a `dict` keyed by parameter identity: `self.state[p] = {'step': 0, 'exp_avg': ..., 'exp_avg_sq': ...}`. The second-moment EMA `v` lives there as `exp_avg_sq`. This indirection — instead of a parallel list `self.v = [...]` like cx29 — is what makes optimizers checkpointable (`optimizer.state_dict()` serialises the WHOLE per-param state dict) and gives them the ability to lazily allocate state on first use.

**The two atoms.**
- **optimizer-state-tensor-buffers** — `self.state[p] = {'exp_avg_sq': t.zeros_like(p), ...}` (lazy or eager). State allocation happens once per parameter, then is reused.
- **ema-second-moment** — the second-moment EMA update on whatever buffer the state dict holds: `state['exp_avg_sq'].mul_(beta2).addcmul_(g, g, value=1-beta2)`.

**Anatomy.**
```python
class StatefulRMSProp:
    def __init__(self, params, lr, beta2, eps):
        self.params = list(params)
        self.lr, self.beta2, self.eps = lr, beta2, eps
        self.state = {}                              # optimizer-state-tensor-buffers.
    @t.inference_mode()
    def step(self):
        for p in self.params:
            if p not in self.state:
                # Lazy allocation — first time we see this param.
                self.state[p] = {'step': 0, 'exp_avg_sq': t.zeros_like(p)}
            s = self.state[p]
            s['step'] += 1
            g = p.grad
            s['exp_avg_sq'].mul_(self.beta2).addcmul_(g, g, value=1-self.beta2)  # ema-second.
            v_hat = s['exp_avg_sq'] / (1 - self.beta2 ** s['step'])
            p.data.addcdiv_(g, v_hat.sqrt().add_(self.eps), value=-self.lr)
```

**Why care.** Loading a checkpoint into a fresh optimizer relies on `state[p]` shape and key conventions. PyTorch's `Adam.state_dict()` returns exactly this nested dict (keyed by *index*, then reattached to params on load).

### Composite Exercise — v EMA stored per-param in a state dict (PyTorch optimizer convention)

**Atoms exercised together**: `ema-second-moment`, `optimizer-state-tensor-buffers`

Implement `cx30_make_stateful_rmsprop()` — return the `StatefulRMSProp` class.

Required structure:
- `__init__(self, params, lr=1e-2, beta2=0.999, eps=1e-8)`:
  - `self.params = list(params)`, `self.lr, self.beta2, self.eps = lr, beta2, eps`.
  - `self.state = {}` — empty dict, keyed by parameter `Tensor` identity.
- `step(self)` — runs under `t.inference_mode()`. For each `p` in `self.params`:
  - If `p not in self.state`, allocate the state lazily: `self.state[p] = {'step': 0, 'exp_avg_sq': t.zeros_like(p)}`.
  - `s = self.state[p]`; `s['step'] += 1`.
  - Update `s['exp_avg_sq']` IN PLACE: `s['exp_avg_sq'].mul_(beta2).addcmul_(g, g, value=1-beta2)` (atom: ema-second-moment).
  - `v_hat = s['exp_avg_sq'] / (1 - beta2 ** s['step'])`.
  - `p.data.addcdiv_(g, v_hat.sqrt().add_(eps), value=-lr)`.
- `zero_grad(self)` — sets each `p.grad = None`.

The test verifies the state-dict atom is present:
- Before first `.step()`, `self.state == {}`.
- After first `.step()` with one param, `len(self.state) == 1` and the key is the parameter itself (by identity).
- `state[p]['exp_avg_sq']` is the SAME tensor across steps (id and data_ptr preserved).
- A second parameter, added later, lazily allocates its OWN state entry on first step.
- The state-stored `exp_avg_sq` evolves under the EMA recurrence (atom: ema-second-moment).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx30_make_stateful_rmsprop():
    """Return the StatefulRMSProp class."""
    raise NotImplementedError

def _test_cx30():
    StatefulRMSProp = cx30_make_stateful_rmsprop()

    # Case A: state is empty pre-step; populated lazily on first step.
    t.manual_seed(0)
    p = t.nn.Parameter(t.randn(3, 4))
    opt = StatefulRMSProp([p], lr=1e-2, beta2=0.9, eps=1e-8)
    assert hasattr(opt, 'state') and isinstance(opt.state, dict)
    assert len(opt.state) == 0, f'state must be empty before any .step(); got {len(opt.state)}'

    p.grad = t.ones_like(p)
    opt.step()
    assert len(opt.state) == 1, f'after 1 step state should have 1 entry; got {len(opt.state)}'
    assert p in opt.state, 'state must be keyed by the Parameter tensor identity'
    s = opt.state[p]
    assert isinstance(s, dict), f'state[p] must be a dict; got {type(s).__name__}'
    assert 'exp_avg_sq' in s, f"state[p] must have key 'exp_avg_sq'; got {list(s.keys())}"
    assert 'step' in s, f"state[p] must have key 'step'; got {list(s.keys())}"
    assert s['step'] == 1

    # Case B: exp_avg_sq is the SAME tensor across steps (in-place updates).
    v_id_before = id(s['exp_avg_sq'])
    v_ptr_before = s['exp_avg_sq'].data_ptr()
    p.grad = t.ones_like(p)
    opt.step()
    s = opt.state[p]  # re-fetch — same dict.
    assert id(s['exp_avg_sq']) == v_id_before, 'exp_avg_sq was rebound across steps'
    assert s['exp_avg_sq'].data_ptr() == v_ptr_before, 'exp_avg_sq storage changed'
    assert s['step'] == 2

    # Case C: numerical EMA trajectory matches manual recurrence.
    # With grad=1 across all steps, beta2=0.9: v0=0, v1=0.1, v2=0.09+0.1=0.19, v3=0.171+0.1=0.271.
    p.grad = t.ones_like(p)
    opt.step()
    assert t.allclose(opt.state[p]['exp_avg_sq'], 0.271 * t.ones_like(p), atol=1e-6), (
        f'EMA trajectory wrong after 3 steps with g=1, beta2=0.9; '
        f'expected 0.271, got {opt.state[p]["exp_avg_sq"][0,0].item():.4f}'
    )

    # Case D: lazy allocation — adding a NEW param later gets its own entry on first step.
    p2 = t.nn.Parameter(t.zeros(5))
    opt.params.append(p2)
    p2.grad = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
    p.grad = t.ones_like(p)
    opt.step()
    assert p2 in opt.state, 'new param must lazily allocate state on first step'
    assert opt.state[p2]['step'] == 1, 'new param state starts at step=1, not 4'
    # v for p2 after 1 step: 0.9*0 + 0.1*g^2 = 0.1 * [1,4,9,16,25] = [0.1,0.4,0.9,1.6,2.5].
    expected_v_p2 = t.tensor([0.1, 0.4, 0.9, 1.6, 2.5])
    assert t.allclose(opt.state[p2]['exp_avg_sq'], expected_v_p2, atol=1e-7)

    # Case E: zero_grad sets all grads to None.
    opt.zero_grad()
    assert p.grad is None and p2.grad is None
    _dd_passed.add('cx30')

_test_cx30()

<details><summary>Show solution — cx30</summary>

```python
def cx30_make_stateful_rmsprop():
    class StatefulRMSProp:
        def __init__(self, params, lr=1e-2, beta2=0.999, eps=1e-8):
            self.params = list(params)
            self.lr = lr
            self.beta2 = beta2
            self.eps = eps
            # Atom B (optimizer-state-tensor-buffers): empty per-param state dict;
            # buffers are lazily allocated on first .step().
            self.state = {}

        @t.inference_mode()
        def step(self):
            for p in self.params:
                if p not in self.state:
                    # Lazy alloc: zeros_like(p) gives a buffer with the right shape/dtype/device.
                    self.state[p] = {'step': 0, 'exp_avg_sq': t.zeros_like(p)}
                s = self.state[p]
                s['step'] += 1
                g = p.grad
                # Atom A (ema-second-moment): in-place EMA update on the dict-stored buffer.
                s['exp_avg_sq'].mul_(self.beta2).addcmul_(g, g, value=1 - self.beta2)
                v_hat = s['exp_avg_sq'] / (1 - self.beta2 ** s['step'])
                p.data.addcdiv_(g, v_hat.sqrt().add_(self.eps), value=-self.lr)

        def zero_grad(self):
            for p in self.params:
                p.grad = None

    return StatefulRMSProp
```

Keying `state` by the Parameter itself (not its index) means moving / reassigning a param list does NOT lose state — the same tensor identity carries through. PyTorch's `Optimizer` does this with `param_groups[i]['params'][j]` as the canonical iteration order, but stores `state` keyed by Tensor. Lazy allocation (`if p not in self.state`) is what lets `torch.optim.Adam` start with no state on construction and grow it on demand — important because `zeros_like(p)` has to know the device/dtype, which is easier after `.to(device)` has been called.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx30'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx30',
        'subtopics': ["Optimizer: Adam EMA second moment", "Optimizer: Per-param state buffers"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()